In [4]:
import json
from collections import defaultdict

**1. Merge Tweet data to make instances longer than 500 characters**

In [ ]:
# Import File
with open("CROSSNEWS_silver.json", "r", encoding = "utf-8") as f:
    data = json.load(f)

# Get Tweets by author
tweets_by_author = defaultdict(list)
for item in data:
    if item["genre"].lower() == "tweet":
        tweets_by_author[item["author"].strip().lower()].append(item["text"].strip())

# Generate 500+ character Tweet
merged_tweets = []
min_char_length = 500

for author, tweets in tweets_by_author.items():
    merged_text = ""
    i = 0
    while i < len(tweets):
        merged_text = tweets[i]
        i += 1
        while len(merged_text) < min_char_length and i < len(tweets):
            merged_text += " " + tweets[i]
            i += 1
        if len(merged_text) >= min_char_length:
            merged_tweets.append({
                "author": author,
                "text": merged_text,
                "genre": "Tweet_Merged"
            })

# Save
with open("merged_tweets.json", "w", encoding = "utf-8") as f:
    json.dump(merged_tweets, f, ensure_ascii = False, indent = 2)

print(f"Merged Tweets: {len(merged_tweets)}")

Merged Tweets: 265016


**2. Build Datasets for experiment** <br>
Generate Positive (Same author) / Negative (Different Author) Pairs

In [2]:
import random
from sklearn.model_selection import train_test_split

In [3]:
import json
import random
from collections import defaultdict

random.seed(7)

# Load original data
with open("crossnews_silver.json", "r", encoding="utf-8") as f:
    original_data = json.load(f)

with open("merged_tweets.json", "r", encoding="utf-8") as f:
    merged_data = json.load(f)

# Separate articles and combine
articles = [item for item in original_data if item["genre"] == "Article"]
combined_data = articles + merged_data

# Group by author
articles_by_author = defaultdict(list)
tweets_by_author = defaultdict(list)

for item in combined_data:
    author = item["author"].strip().lower()
    text = item["text"].strip()
    genre = item["genre"]

    if genre == "Article":
        articles_by_author[author].append(text)
    elif genre == "Tweet_Merged":
        tweets_by_author[author].append(text)

# Only use authors who have both articles and tweets
valid_authors = list(set(articles_by_author) & set(tweets_by_author))

positive_pairs = []
negative_pairs = []
used_articles = set()
used_tweets = set()

for author in valid_authors:
    arts = [a for a in articles_by_author[author] if a not in used_articles]
    twts = [t for t in tweets_by_author[author] if t not in used_tweets]

    if not arts or not twts:
        continue

    # Sample 1 unused article and 1 unused tweet
    art = random.choice(arts)
    twt = random.choice(twts)

    # Add to positive pair
    positive_pairs.append({
        "text1": art,
        "text2": twt,
        "label": 1
    })

    used_articles.add(art)
    used_tweets.add(twt)

    # Make a negative pair: article from this author + tweet from another author
    other_authors = [a for a in valid_authors if a != author and tweets_by_author[a]]
    if not other_authors:
        continue

    neg_twt_author = random.choice(other_authors)
    twt_neg_candidates = [t for t in tweets_by_author[neg_twt_author] if t not in used_tweets]
    if not twt_neg_candidates:
        continue

    neg_twt = random.choice(twt_neg_candidates)

    negative_pairs.append({
        "text1": art,
        "text2": neg_twt,
        "label": 0
    })

    used_tweets.add(neg_twt)

# Combine and shuffle
all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)

# Save
with open("av_pairs_balanced_sample.json", "w", encoding="utf-8") as f:
    json.dump(all_pairs, f, indent=2, ensure_ascii=False)

print(f"Total authors used: {len(positive_pairs)}")
print(f"Total pairs generated: {len(all_pairs)} (Positive: {len(positive_pairs)}, Negative: {len(negative_pairs)})")

Total authors used: 2222
Total pairs generated: 4425 (Positive: 2222, Negative: 2203)


In [5]:
# split the dataset into training/validation
from sklearn.model_selection import train_test_split

with open("av_pairs_balanced_sample.json", "r", encoding = "utf-8") as f:
    all_data = json.load(f)

train_data, val_data = train_test_split(all_data, test_size = 0.2, random_state = 7)

with open("train.json", "w", encoding = "utf-8") as f:
    json.dump(train_data, f, indent = 2, ensure_ascii = False)

with open("val.json", "w", encoding = "utf-8") as f:
    json.dump(val_data, f, indent = 2, ensure_ascii = False)

In [1]:
# Define the Dataset class

from torch.utils.data import Dataset
from transformers import AutoTokenizer

class AVPairDataset(Dataset):
    def __init__(self, json_path, tokenizer_name="roberta-base", max_length=512):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoded = self.tokenizer(
            item["text1"],
            item["text2"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": int(item["label"])
        }

In [6]:
# make dataset instances
train_dataset = AVPairDataset("train.json")
val_dataset = AVPairDataset("val.json")

In [7]:
# Load the Huggingface model and Tokeniser 
from transformers import AutoModelForSequenceClassification, trainer

model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels = 2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",
    evaluation_strategy = "epoch",
    save_strategy = "no",
    num_train_epochs = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    logging_steps = 50,
    report_to = "none"
)

/opt/miniconda3/envs/com6018/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
from transformers import Trainer
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.697900,0.694538
2,0.706300,0.694415
3,0.691800,0.693180


TrainOutput(global_step=1329, training_loss=0.700963498955835, metrics={'train_runtime': 68047.6046, 'train_samples_per_second': 0.156, 'train_steps_per_second': 0.02, 'total_flos': 2794239407923200.0, 'train_loss': 0.700963498955835, 'epoch': 3.0})

In [11]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"]
    }

python(38550) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import evaluate
import numpy as np

# metric 
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"]
    }

# re-define the trainer with metrics
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.evaluate()

{'eval_loss': 0.6931799054145813,
 'eval_model_preparation_time': 0.0023,
 'eval_accuracy': 0.5016949152542373,
 'eval_f1': 0.6681715575620768,
 'eval_runtime': 813.5514,
 'eval_samples_per_second': 1.088,
 'eval_steps_per_second': 0.136}